# Robust MPC Main Workspace

This notebook is the main workspace for the project.

Use it to:

- choose simulation parameters
- set start pose, end pose, constraints, and disturbance levels
- generate video and plots for a run
- inspect trajectories and quick health metrics
- run lightweight sanity tests after changes

The Python files under `src/` now act as the reusable inner engine.

## Setup

Run this first so the notebook can import the refactored modules from the repository root.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, Markdown, Video, display

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.simulation import SimulationConfig, run_simulation

PROJECT_ROOT

## Editable run parameters

Edit this cell before each experiment. It is the main control panel for the simulation.

In [ ]:
RUN_NAME = "baseline_workspace_run"

START_POSE = [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
END_POSE = [-1.0, -0.8, 2.0, 2.0, 2.7, 1.0]

SIMULATION_PARAMS = {
    "duration": 1.5,
    "horizon": 10,
    "framerate": 60,
    "seed": 7,
}

CONSTRAINT_PARAMS = {
    "velocity_limit": 10.0,
    "control_limit": 100.0,
}

DISTURBANCE_PARAMS = {
    "disturbance_bound": 1.0,
    "tube_disturbance_bound": 0.1,
}

COST_PARAMS = {
    "position_weight": 200.0,
    "velocity_weight": 20.0,
    "control_weight": 0.005,
    "terminal_multiplier": 200.0,
}

GENERATE_VIDEO = True
GENERATE_PLOTS = True

OUTPUT_DIR = PROJECT_ROOT / "artifacts" / RUN_NAME

## Build config helpers

These helpers turn the editable parameter cell into a `SimulationConfig`, run the simulation, and display the outputs.

In [ ]:
def build_workspace_config() -> SimulationConfig:
    start_pose = np.asarray(START_POSE, dtype=float)
    end_pose = np.asarray(END_POSE, dtype=float)

    if start_pose.shape != end_pose.shape:
        raise ValueError("START_POSE and END_POSE must have the same shape.")

    if start_pose.ndim != 1:
        raise ValueError("START_POSE and END_POSE must be 1D vectors.")

    config_kwargs = {
        "output_dir": OUTPUT_DIR,
        "initial_positions": start_pose,
        "target_positions": end_pose,
        "render_video": GENERATE_VIDEO,
        "save_plots": GENERATE_PLOTS,
        **SIMULATION_PARAMS,
        **CONSTRAINT_PARAMS,
        **DISTURBANCE_PARAMS,
        **COST_PARAMS,
    }
    return SimulationConfig(**config_kwargs)


def config_to_display_dict(config: SimulationConfig) -> dict:
    serializable = {}
    for key, value in config.__dict__.items():
        if isinstance(value, Path):
            serializable[key] = str(value)
        elif isinstance(value, np.ndarray):
            serializable[key] = value.tolist()
        else:
            serializable[key] = value
    return serializable


def summarize_result(result) -> dict:
    final_error = result.joint_positions[-1] - result.target_positions
    return {
        "steps": int(result.time_steps.shape[0]),
        "infeasible_steps": int(result.infeasible_steps.sum()),
        "final_error_norm": float(np.linalg.norm(final_error)),
        "max_abs_control": float(np.abs(result.control_inputs).max()),
        "video_path": None if result.video_path is None else str(result.video_path),
        "plot_paths": {key: str(path) for key, path in result.plot_paths.items()},
    }


def plot_result_inline(result) -> None:
    fig, axes = plt.subplots(3, 1, figsize=(11, 12), sharex=True)

    for joint_index in range(result.joint_positions.shape[1]):
        axes[0].plot(result.time_steps, result.joint_positions[:, joint_index], label=f"Joint {joint_index + 1}")
        axes[0].axhline(result.target_positions[joint_index], color="black", linestyle="--", alpha=0.2)

    errors = result.joint_positions - result.target_positions[None, :]
    for joint_index in range(errors.shape[1]):
        axes[1].plot(result.time_steps, errors[:, joint_index], label=f"Error {joint_index + 1}")
        axes[1].fill_between(
            result.time_steps,
            -result.tube_bounds[joint_index],
            result.tube_bounds[joint_index],
            alpha=0.08,
            color="green",
        )

    for joint_index in range(result.control_inputs.shape[1]):
        axes[2].plot(result.time_steps, result.control_inputs[:, joint_index], label=f"u {joint_index + 1}")

    axes[0].set_title("Joint positions")
    axes[0].set_ylabel("Position (rad)")
    axes[0].grid(True)
    axes[0].legend(loc="upper right", ncol=2)

    axes[1].set_title("Tracking error and effective tube bounds")
    axes[1].set_ylabel("Error (rad)")
    axes[1].grid(True)

    axes[2].set_title("Control inputs")
    axes[2].set_ylabel("Acceleration-like input")
    axes[2].set_xlabel("Time (s)")
    axes[2].grid(True)

    plt.tight_layout()
    plt.show()


def display_saved_artifacts(result) -> None:
    if result.video_path is not None and result.video_path.exists():
        display(Markdown("### Simulation video"))
        display(Video(str(result.video_path), embed=True))

    if result.plot_paths:
        display(Markdown("### Saved plots"))
        for label, path in result.plot_paths.items():
            if path.exists():
                display(Markdown(f"**{label.replace('_', ' ').title()}**  \n`{path}`"))
                display(Image(filename=str(path)))


## Preview the current configuration

Run this before launching a simulation to verify the chosen parameters.

In [ ]:
workspace_config = build_workspace_config()
print(json.dumps(config_to_display_dict(workspace_config), indent=2))

## Run the simulation

This is the main execution cell. It uses the chosen parameters, generates the video if enabled, saves plots if enabled, and returns the result object for further analysis.

In [ ]:
workspace_config = build_workspace_config()
workspace_result = run_simulation(workspace_config)

summary = summarize_result(workspace_result)
print(json.dumps(summary, indent=2))

## Inspect outputs

The first function draws inline plots directly from the returned arrays. The second function displays the saved video and saved plot files generated by the simulation.

In [ ]:
plot_result_inline(workspace_result)
display_saved_artifacts(workspace_result)

## Parameter sweep scratchpad

Use this cell when you want to compare a few settings quickly without editing the engine code in `src/`.

In [ ]:
candidate_horizons = [5, 10, 15]
sweep_rows = []

for horizon in candidate_horizons:
    sweep_config = SimulationConfig(
        **{
            **build_workspace_config().__dict__,
            "horizon": horizon,
            "render_video": False,
            "save_plots": False,
            "output_dir": OUTPUT_DIR / f"horizon-{horizon}",
        }
    )
    sweep_result = run_simulation(sweep_config)
    sweep_rows.append(
        {
            "horizon": horizon,
            "steps": int(sweep_result.time_steps.shape[0]),
            "infeasible_steps": int(sweep_result.infeasible_steps.sum()),
            "final_error_norm": float(np.linalg.norm(sweep_result.joint_positions[-1] - sweep_result.target_positions)),
            "max_abs_control": float(np.abs(sweep_result.control_inputs).max()),
        }
    )

sweep_rows

## Sanity tests

These are lightweight checks for the current workspace settings and for a very short fast run.

In [ ]:
def run_sanity_checks(duration: float = 0.05) -> None:
    config = SimulationConfig(
        **{
            **build_workspace_config().__dict__,
            "duration": duration,
            "render_video": False,
            "save_plots": False,
            "output_dir": OUTPUT_DIR / "sanity-checks",
        }
    )
    result = run_simulation(config)

    assert result.time_steps.ndim == 1
    assert result.time_steps.shape[0] > 0
    assert result.joint_positions.shape[0] == result.time_steps.shape[0]
    assert result.control_inputs.shape[0] == result.time_steps.shape[0]
    assert result.joint_positions.shape[1] == result.target_positions.shape[0]
    assert result.control_inputs.shape[1] == result.target_positions.shape[0]
    assert result.infeasible_steps.shape[0] == result.time_steps.shape[0]
    assert np.isfinite(result.joint_positions).all()
    assert np.isfinite(result.control_inputs).all()
    assert np.isfinite(result.tube_bounds).all()

    print("Sanity checks passed.")
    print(f"Steps: {result.time_steps.shape[0]}")
    print(f"Infeasible steps: {int(result.infeasible_steps.sum())}")
    print(f"Final error norm: {np.linalg.norm(result.joint_positions[-1] - result.target_positions):.4f}")


run_sanity_checks()

## Future work

- Add a richer comparison table for multiple controller settings.
- Track settling time, overshoot, and energy-like control cost per run.
- Add disturbance sweeps and robustness summaries across random seeds.
- Compare the current MPC setup with unconstrained MPC and simpler PD baselines.
- Replace the current linear prediction model with a more expressive dynamics approximation.
- Add dedicated experiment sections for obstacle-aware motion or task-space goals.
